In [1]:
!git clone https://github.com/Wenhao-Sun77/Just-in-Time.git
%cd Just-in-Time
!pip install diffusers==0.37.0 transformers==4.55.0 accelerate bitsandbytes sentencepiece scipy==1.16.0

Cloning into 'Just-in-Time'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 39 (delta 9), reused 14 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 5.83 MiB | 25.52 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/Just-in-Time
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 55.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 79.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 56.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: scipy
    F

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN successfully retrieved and set as environment variable.")
except Exception as e:
    print(f"Failed to retrieve HF_TOKEN: {e}")

HF_TOKEN successfully retrieved and set as environment variable.


In [3]:
import sys
import gc
import time
import torch
from transformers import BitsAndBytesConfig, T5EncoderModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig, FluxTransformer2DModel
from diffusers.models.modeling_utils import ModelMixin
from transformers.modeling_utils import PreTrainedModel
from accelerate.hooks import add_hook_to_module, AlignDevicesHook

# Bugfix 2: Diffusers Patch for 8-bit .to() crashes
original_to = ModelMixin.to
def safe_to(self, *args, **kwargs):
    if getattr(self, "is_loaded_in_8bit", False) or getattr(self, "is_loaded_in_4bit", False):
        return self
    return original_to(self, *args, **kwargs)
ModelMixin.to = safe_to

# Bugfix 2: Transformers Patch for 8-bit .to() crashes
original_tf_to = PreTrainedModel.to
def safe_tf_to(self, *args, **kwargs):
    if getattr(self, "is_loaded_in_8bit", False) or getattr(self, "is_loaded_in_4bit", False):
        return self
    return original_tf_to(self, *args, **kwargs)
PreTrainedModel.to = safe_tf_to

# Append the flux directory to sys.path
sys.path.append("./flux")
from pipeline_flux_JiT import FluxPipeline_JiT

# Custom Multi-GPU Device Map definition (for logical reference)
custom_device_map = {
    "text_encoder": 1,
    "text_encoder_2": 1,
    "tokenizer": 1,
    "tokenizer_2": 1,
    "vae": 0,
    "transformer": 0
}

# Bugfix 1: Manual Component Isolation to prevent Disk Memory Miscalculation (CPU offload crash)
text_encoder_2 = T5EncoderModel.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    subfolder="text_encoder_2",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map={"": custom_device_map["text_encoder_2"]},
    torch_dtype=torch.float16
)

transformer = FluxTransformer2DModel.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    subfolder="transformer",
    quantization_config=DiffusersBitsAndBytesConfig(load_in_8bit=True),
    device_map={"": custom_device_map["transformer"]},
    torch_dtype=torch.float16
)

# Initialize pipeline strictly using FluxPipeline_JiT.from_pretrained 
# (Note: Diffusers pipelines strictly expect strings for device_map, so we omit custom_device_map here)
pipeline = FluxPipeline_JiT.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    text_encoder_2=text_encoder_2,
    transformer=transformer,
    torch_dtype=torch.float16
)

# Sync the pipeline's execution device (This moves the unquantized VAE and text_encoder to cuda:0)
pipeline.to("cuda:0")

# Explicitly enforce the custom device map for the small float16 components to honor the dictionary request
pipeline.text_encoder.to(f"cuda:{custom_device_map['text_encoder']}")

# Bugfix 5: VAE Spatial Tiling to prevent Decode OOM
pipeline.enable_vae_tiling()
pipeline.enable_vae_slicing()

# Bugfix 3 & 4: Attach hardware isolation hooks for tensor routing (avoids monkey-patching forward pass)
# NOTE: We DO NOT attach a hook to the VAE because we explicitly move the VAE to cuda:1 in the generation loop. 
# If we attached a hook here, it would permanently lock the VAE weights to cuda:0 and crash the manual decode.
add_hook_to_module(pipeline.text_encoder, AlignDevicesHook(execution_device=custom_device_map["text_encoder"]))
add_hook_to_module(pipeline.text_encoder_2, AlignDevicesHook(execution_device=custom_device_map["text_encoder_2"]))
add_hook_to_module(pipeline.transformer, AlignDevicesHook(execution_device=custom_device_map["transformer"]))

# Bugfix 7: Override Diffusers naive _execution_device property. 
# Because text_encoder has a hook for cuda:1, the pipeline mistakenly infers the base device as cuda:1.
# We must force the base device to cuda:0 so all pipeline-level latents and JiT interpolation grids match the DiT's device.
type(pipeline)._execution_device = property(lambda self: torch.device("cuda:0"))


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

text_encoder_2/model-00001-of-00002.safe(…):   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/flux/pipeline_flux.py:577: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `FluxPipeline_JiT` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/flux/pipeline_flux.py:550: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `FluxPipeline_JiT` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [4]:
pipeline.set_params(
    total_steps=18,
    stage_ratios=[0.4, 0.65, 1.0],
    sparsity_ratios=[0.35, 0.62, 1.0],
    use_checkerboard_init=True,
    use_adaptive=True,
    use_beta_sigmas=True,
    alpha=1.4,
    beta=0.425,
    microflow_relax_steps=3
)

JiT Configuration:
  Total steps: 18
  Stage divisions: [7, 11, 18]
  Token densities: [0.35, 0.62, 1.0]
  Adaptive densification: True


In [5]:
prompts = [
    "A grand piano made entirely of transparent, crystal-clear ice, with delicate frost patterns on its surface. It sits in a warm, sunlit concert hall, slowly melting, with water dripping onto the polished wooden floor. Photorealistic, poignant.",
    "A golden retriever dog playing on vibrant green grass, cinematic lighting, highly detailed, 4k resolution.",
    "A glowing neon sign in a dark, rainy cyberpunk alleyway that reads 'SYSTEMS' in bright pink letters, reflecting on a wet cobblestone street.",
    "A red coffee mug sitting on the left side of a wooden desk, next to a stack of three hardback books on the right, with a sleeping tabby cat curled up behind them",
    "A close-up, highly detailed portrait of an elderly fisherman with deep facial wrinkles, a white beard, wearing a yellow raincoat, looking directly into the camera."
]
seeds = [42, 1024, 2048, 777, 999]

# Clean up progress bar cascade
pipeline.set_progress_bar_config(leave=False)

for i, (prompt, seed) in enumerate(zip(prompts, seeds)):
    print(f"\nGenerating Image {i+1}: {prompt[:50]}...")
    generator = torch.Generator(device="cuda:0").manual_seed(seed)
    
    start_time = time.perf_counter()
    
    # Instruct the pipeline to return UNDECODED latents to avoid GPU 0 OOM spike
    output = pipeline(
        prompt=prompt,
        height=1024,
        width=1024,
        guidance_scale=3.5,
        max_sequence_length=256,
        generator=generator,
        output_type="latent"
    )
    
    # Forcefully purge CUDA memory on GPU 0
    torch.cuda.empty_cache()
    gc.collect()
    
    # Safely shuttle latents to GPU 1 and decode with zero memory pressure
    latents = output.images[0] if isinstance(output.images, list) else output.images
    
    H_packed = 1024 // 16
    W_packed = 1024 // 16
    latents = pipeline._unpack_latents(latents, H_packed, W_packed)
    latents = (latents / pipeline.vae.config.scaling_factor) + pipeline.vae.config.shift_factor
    
    # Bugfix 8: Force the VAE completely onto cuda:1 right before decode 
    # to bypass any pipeline reversions that might have happened during __call__
    pipeline.vae.to("cuda:1")
    with torch.no_grad():
        image = pipeline.vae.decode(latents.to("cuda:1"), return_dict=False)[0]
        
    image = pipeline.image_processor.postprocess(image, output_type="pil")[0]
    
    end_time = time.perf_counter()
    print(f"Image {i+1} completed in {end_time - start_time:.2f} seconds.")
    
    image.save(f"JiT_Run_{i}.png")



Generating Image 1: A grand piano made entirely of transparent, crysta...

JiT: Training-Free Spatial Acceleration for DiTs
Image resolution: 1024×1024
VAE latent: 128×128
Packed tokens: 64×64 (4096 tokens)
Token densities: [0.35, 0.62, 1.0]
Adaptive densification: True

Stage 2: Initialized 1433 tokens (density=35.00%)


Denoising:  39%|███▉      | 7/18 [00:10<00:15,  1.44s/it, stage=2, active=1433, total=4096, ratio=35.0%]


Step 7: Stage transition 2 → 1
  Using adaptive densification (variance-based)
  Newly activated tokens: 1106


Denoising:  61%|██████    | 11/18 [00:20<00:15,  2.20s/it, stage=1, active=2539, total=4096, ratio=62.0%]


Step 11: Stage transition 1 → 0
  Using adaptive densification (variance-based)
  Newly activated tokens: 1557


...

Denoising:  39%|███▉      | 7/18 [00:10<00:16,  1.50s/it, stage=2, active=1433, total=4096, ratio=35.0%]


Step 7: Stage transition 2 → 1
  Using adaptive densification (variance-based)
  Newly activated tokens: 1106


Denoising:  61%|██████    | 11/18 [00:20<00:15,  2.26s/it, stage=1, active=2539, total=4096, ratio=62.0%]


Step 11: Stage transition 1 → 0
  Using adaptive densification (variance-based)
  Newly activated tokens: 1557



Generation complete
Final active tokens: 4096/4096 (100.00%)

Image 5 completed in 51.35 seconds.
